## PersonalityAI: Discovering Human Archetypes through Unsupervised Learning

## 1. 💼 Business Problem
----
- Companies struggle to understand the behavioral diversity of their users or employees.
  
- Standard demographic data (like age or location) doesn't tell you how a person thinks, works, or prefers to interact.

**The Problem:** How can we automatically discover hidden, meaningful human personality profiles from survey responses without relying on manual, subjective categorizations?

## 2. 🎯 Business Value
-----

- By running Unsupervised Clustering (K-Means) on this survey data, we can build a system that delivers value across multiple industries:
  
  - **HR & Recruitment:** Match job candidates with team cultures or specific roles (e.g., placing high-conscientiousness individuals in detail-oriented roles).
    
  - **Personalized Marketing & UX:** Adapt app onboarding, recommendations, and messaging tones based on user behavioral clusters (e.g., tailoring Spotify homepages for outgoing vs. reflective listeners).
  - **Product Personalization:** Recommend features, courses, or content that naturally align with how an individual processes information.

## 3. Loading the dataset
----

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load dataset (Kaggle's file uses tab separation '\t')
df = pd.read_csv("C:\\Users\\hp\\Downloads\\data-final.csv\\data-final.csv", sep='\t' , nrows=100000)


df.head()


,EXT1,EXT2,EXT3,EXT4,EXT5,EXT6,EXT7,EXT8,EXT9,EXT10,...,dateload,screenw,screenh,introelapse,testelapse,endelapse,IPC,country,lat_appx_lots_of_err,long_appx_lots_of_err
0,4.0,1.0,5.0,2.0,5.0,1.0,5.0,2.0,4.0,1.0,...,2016-03-03 02:01:01,768,1024,9,234.0,6,1,GB,51.5448,0.1991
1,3.0,5.0,3.0,4.0,3.0,3.0,2.0,5.0,1.0,5.0,...,2016-03-03 02:01:20,1360,768,12,179.0,11,1,MY,3.1698,101.706
2,2.0,3.0,4.0,4.0,3.0,2.0,1.0,3.0,2.0,5.0,...,2016-03-03 02:01:56,1366,768,3,186.0,7,1,GB,54.9119,-1.3833
3,2.0,2.0,2.0,3.0,4.0,2.0,2.0,4.0,1.0,4.0,...,2016-03-03 02:02:02,1920,1200,186,219.0,7,1,GB,51.75,-1.25
4,3.0,3.0,3.0,3.0,5.0,3.0,3.0,5.0,3.0,4.0,...,2016-03-03 02:02:57,1366,768,8,315.0,17,2,KE,1.0,38.0


In [26]:
# 2. Inspect shape & initial columns
df.shape

(1015341, 110)

## 4. Defining the Big Five (OCEAN) Personality Traits
-----

To build our model, we must map raw survey answers to these 5 core human traits. Every individual receives a score on a scale of 1 (Very Low) to 5 (Very High) for each dimension:

- **Openness (OPN) — Creativity & Intellectual Curiosity**
  
  - High (4–5): Highly imaginative, open to novel experiences, artistic, and intellectually curious.
  - Low (1–2): Practical, conventional, prefers routine, and focuses on concrete facts over abstract ideas.
    ```
    ```
- **Conscientiousness (CSN) — Organization & Goal-Directed Behavior**
  
  - High (4–5): Methodical, highly organized, punctual, reliable, and driven by self-discipline.
  - Low (1–2): Spontaneous, flexible, easygoing with rules, but prone to procrastination or disorder.
    ```
    ```
- **Extraversion (EXT) — Social Battery & Expressiveness**
  - High (4–5): Outgoing, high-energy, thrives in groups, and comfortable taking center stage.
  - Low (1–2): Reserved, reflective, prefers quiet settings, and recharges through solitude (introverted).
    ```
    ``` 
- **Agreeableness (AGR) — Empathy & Social Harmony**
  - High (4–5): Compassionate, cooperative, trusting, empathetic, and values team cohesion.
  - Low (1–2): Skeptical, competitive, direct, blunt, and willing to challenge or confront others.
    ```
    ```
- **Neuroticism / Emotional Stability (EST) — Stress Reactivity & Sensitivity**
  - High (4–5): Sensitive to stress, experiences frequent anxiety or mood swings, and easily overwhelmed.
  - Low (1–2): Calm, resilient, emotionally steady, and relaxed under pressure.
    ```
    ```
 
> Our Data Science Objective: Aggregate the 50 raw survey answers into these 5 clean trait scores so that machine learning algorithms can segment individuals and make predictions.

## 5. Exploratory Data Analysis and Data Cleaning
----

- This dataset contains over 110 columns, but not every column is needed for building our personality model. Many columns contain technical metadata (such as country codes, IP addresses, and screen sizes).
  
- Our goal is to analyze human personality using the Big Five (OCEAN) framework and eventually group similar people into distinct personality profiles/clusters using Machine Learning.
- To do this, each person will be evaluated across all 5 dimensions (Openness, Conscientiousness, Extraversion, Agreeableness, and Neuroticism). Therefore, we only need the 50 question columns (EXT1–OPN10) that capture responses to the survey.
- All non-question columns will be filtered out to keep our dataset clean and efficient.

In [57]:
list(df.columns)

['EXT1',
 'EXT2',
 'EXT3',
 'EXT4',
 'EXT5',
 'EXT6',
 'EXT7',
 'EXT8',
 'EXT9',
 'EXT10',
 'EST1',
 'EST2',
 'EST3',
 'EST4',
 'EST5',
 'EST6',
 'EST7',
 'EST8',
 'EST9',
 'EST10',
 'AGR1',
 'AGR2',
 'AGR3',
 'AGR4',
 'AGR5',
 'AGR6',
 'AGR7',
 'AGR8',
 'AGR9',
 'AGR10',
 'CSN1',
 'CSN2',
 'CSN3',
 'CSN4',
 'CSN5',
 'CSN6',
 'CSN7',
 'CSN8',
 'CSN9',
 'CSN10',
 'OPN1',
 'OPN2',
 'OPN3',
 'OPN4',
 'OPN5',
 'OPN6',
 'OPN7',
 'OPN8',
 'OPN9',
 'OPN10',
 'EXT1_E',
 'EXT2_E',
 'EXT3_E',
 'EXT4_E',
 'EXT5_E',
 'EXT6_E',
 'EXT7_E',
 'EXT8_E',
 'EXT9_E',
 'EXT10_E',
 'EST1_E',
 'EST2_E',
 'EST3_E',
 'EST4_E',
 'EST5_E',
 'EST6_E',
 'EST7_E',
 'EST8_E',
 'EST9_E',
 'EST10_E',
 'AGR1_E',
 'AGR2_E',
 'AGR3_E',
 'AGR4_E',
 'AGR5_E',
 'AGR6_E',
 'AGR7_E',
 'AGR8_E',
 'AGR9_E',
 'AGR10_E',
 'CSN1_E',
 'CSN2_E',
 'CSN3_E',
 'CSN4_E',
 'CSN5_E',
 'CSN6_E',
 'CSN7_E',
 'CSN8_E',
 'CSN9_E',
 'CSN10_E',
 'OPN1_E',
 'OPN2_E',
 'OPN3_E',
 'OPN4_E',
 'OPN5_E',
 'OPN6_E',
 'OPN7_E',
 'OPN8_E',
 'OPN9_E',
 '

### 5.1  Filtering out the columns we need
---

In [127]:
#create an list to store those column

question_columns = []

for trait in ['EXT' , 'EST' , 'AGR' , 'CSN' , 'OPN']:
    for i in range(1 , 11) :
        column_name = f"{trait}{i}"
        question_columns.append(column_name)
        
print(question_columns) 


['EXT1', 'EXT2', 'EXT3', 'EXT4', 'EXT5', 'EXT6', 'EXT7', 'EXT8', 'EXT9', 'EXT10', 'EST1', 'EST2', 'EST3', 'EST4', 'EST5', 'EST6', 'EST7', 'EST8', 'EST9', 'EST10', 'AGR1', 'AGR2', 'AGR3', 'AGR4', 'AGR5', 'AGR6', 'AGR7', 'AGR8', 'AGR9', 'AGR10', 'CSN1', 'CSN2', 'CSN3', 'CSN4', 'CSN5', 'CSN6', 'CSN7', 'CSN8', 'CSN9', 'CSN10', 'OPN1', 'OPN2', 'OPN3', 'OPN4', 'OPN5', 'OPN6', 'OPN7', 'OPN8', 'OPN9', 'OPN10']


### 5.2 Creating a new dataframe of filtered 50 columns
----

In [113]:
# now that we've filtered the 50 columns
# now creating a dataframe like:

df_50 = df[question_columns].copy()
df_50.head()


,EXT1,EXT2,EXT3,EXT4,EXT5,EXT6,EXT7,EXT8,EXT9,EXT10,...,OPN1,OPN2,OPN3,OPN4,OPN5,OPN6,OPN7,OPN8,OPN9,OPN10
0,4.0,1.0,5.0,2.0,5.0,1.0,5.0,2.0,4.0,1.0,...,5.0,1.0,4.0,1.0,4.0,1.0,5.0,3.0,4.0,5.0
1,3.0,5.0,3.0,4.0,3.0,3.0,2.0,5.0,1.0,5.0,...,1.0,2.0,4.0,2.0,3.0,1.0,4.0,2.0,5.0,3.0
2,2.0,3.0,4.0,4.0,3.0,2.0,1.0,3.0,2.0,5.0,...,5.0,1.0,2.0,1.0,4.0,2.0,5.0,3.0,4.0,4.0
3,2.0,2.0,2.0,3.0,4.0,2.0,2.0,4.0,1.0,4.0,...,4.0,2.0,5.0,2.0,3.0,1.0,4.0,4.0,3.0,3.0
4,3.0,3.0,3.0,3.0,5.0,3.0,3.0,5.0,3.0,4.0,...,5.0,1.0,5.0,1.0,5.0,1.0,5.0,3.0,5.0,5.0


### 5.3 Shape of the dataframe
----

In [79]:
df_50.shape

(100000, 50)

- df_50 dataframe has 100000 samples
- and 50 columns

### 5.4 Checking Null and zeroes
----

Spotting both NaN and Zero is important for us here, because:

**1. Zero tricks the math engine** : Computers view zeors as a real and valid number.


- If a scale is 1 to 5, and a user has answers: [5, 5, 5, 5, 0], Python calculates the average as: {5 + 5 + 5 + 5 + 0}\{5} = {4.0}
- The Problem: The 0 drags down their actual average score (which should be 5.0) because the computer thinks 0 is a very low rating, not a skipped question.

- Zero suggests, The form WAS submitted, but a question was empty.  
- What happened: The user might've clicked the "Submit" button at the bottom of the page. The website packaged up all 50 answers and sent them to the database. Because the user skipped Question 3 for instance, the web form sent its default empty value (0) for that specific field.
```
```
**2. NaN breaks the machine learning pipeline** :

- Unlike 0, Python recognizes NaN as missing data. But instead of fixing it, algorithms will literally crash.
  
- 5 + NaN = NaN
- Machine learning tools (like K-Means or PCA) rely on calculating spatial distances. If you pass a NaN into scikit-learn, it throws a ValueError and shuts down because it cannot calculate the distance to "nothing."
- NaN suggests the form was never submitted or the data got lost.
- What happened: The user might've completed 20 questions, got bored, and closed the browser tab. They never clicked "Submit." Or, their internet cut out mid-quiz so nothing was recorded.


In [174]:

total_nans = df_50.isnull().sum().sum()

total_zeroes = (df_50==0).sum().sum()

print("Total Missing Values (NaN):", total_nans)
print("Total Skipped Questions (0s):", total_zeros)

Total Missing Values (NaN): 150
Total Skipped Questions (0s): 40398


**150 NaNs:** Very few people had network crashes or no submissions.

**40,398 0s:** A massive number of people submitted the test while leaving certain questions skipped or untouched.

### 5.5 Data Cleaning
----

In [202]:
# 1. Filter out rows containing 0s (keep rows where ALL 50 answers are non-zero)
df_clean = df_50[(df_50 != 0).all(axis = 1)]

# 2. Drop any remaining NaN rows
df_clean = df_clean.dropna()

df_clean.head()

,EXT1,EXT2,EXT3,EXT4,EXT5,EXT6,EXT7,EXT8,EXT9,EXT10,...,OPN1,OPN2,OPN3,OPN4,OPN5,OPN6,OPN7,OPN8,OPN9,OPN10
0,4.0,1.0,5.0,2.0,5.0,1.0,5.0,2.0,4.0,1.0,...,5.0,1.0,4.0,1.0,4.0,1.0,5.0,3.0,4.0,5.0
1,3.0,5.0,3.0,4.0,3.0,3.0,2.0,5.0,1.0,5.0,...,1.0,2.0,4.0,2.0,3.0,1.0,4.0,2.0,5.0,3.0
2,2.0,3.0,4.0,4.0,3.0,2.0,1.0,3.0,2.0,5.0,...,5.0,1.0,2.0,1.0,4.0,2.0,5.0,3.0,4.0,4.0
3,2.0,2.0,2.0,3.0,4.0,2.0,2.0,4.0,1.0,4.0,...,4.0,2.0,5.0,2.0,3.0,1.0,4.0,4.0,3.0,3.0
4,3.0,3.0,3.0,3.0,5.0,3.0,3.0,5.0,3.0,4.0,...,5.0,1.0,5.0,1.0,5.0,1.0,5.0,3.0,5.0,5.0


In [204]:
# 3. Verify our new dataset shape
print("Original Rows:", df_50.shape[0])
print("Cleaned Rows Remaining:", df_clean.shape[0])
print("Total Rows Removed:", df_50.shape[0] - df_clean.shape[0])

Original Rows: 100000
Cleaned Rows Remaining: 85393
Total Rows Removed: 14607


In [212]:
# Inspect summary statistics for all 50 questions
df_clean.describe().T

,count,mean,std,min,25%,50%,75%,max
EXT1,85393.0,2.660944,1.239866,1.0,2.0,3.0,4.0,5.0
EXT2,85393.0,2.826122,1.307535,1.0,2.0,3.0,4.0,5.0
EXT3,85393.0,3.312086,1.180154,1.0,2.0,3.0,4.0,5.0
EXT4,85393.0,3.182357,1.207267,1.0,2.0,3.0,4.0,5.0
EXT5,85393.0,3.279379,1.233984,1.0,2.0,3.0,4.0,5.0
EXT6,85393.0,2.489829,1.226634,1.0,2.0,2.0,3.0,5.0
EXT7,85393.0,2.802513,1.367995,1.0,2.0,3.0,4.0,5.0
EXT8,85393.0,3.454791,1.240552,1.0,3.0,4.0,5.0,5.0
EXT9,85393.0,3.010001,1.323873,1.0,2.0,3.0,4.0,5.0
EXT10,85393.0,3.618716,1.270636,1.0,3.0,4.0,5.0,5.0


### 5.6 Reverse Scoring
---
- Surveys like this one are designed in such a manner that contains the mix of positive and negative statements to stop people from blindly clicking "5" for every single item without reading.

- For example,  if we look at two real questions from the Extraversion (EXT) section of this dataset:

  **1. EXT1 (Positively Keyed):** "I am the life of the party."
  - **Rating 5:** Extremely High Extraversion.
  - **Rating 1:** Extremely Low Extraversion.
  ```
  ```  
  **2. EXT2 (Negatively Keyed):**"I don't talk a lot."
  - **Rating 5 :** Extremely Low Extraversion (very quiet/introverted)
  - **Rating 1:** Extremely High Extraversion (very talkative)
    ```
    ```
- Imagine a very outgoing person takes this test. They will likely answer:
  - **EXT1 = 5** ("I am the life of the party" - Strongly Agree)
  - **EXT2 = 1** ("I don't talk a lot" - Strongly Disagree)
  - If we simply take the average of these two raw answers:$$\text{Average} = \frac{5 + 1}{2} = 3.0 \quad \text{(Neutral!)}$$
  - The math says this person is completely neutral, even though their answers prove they are extremely extraverted! Because EXT2 measures extraversion in reverse, its scale pulls the score in the wrong direction.
```
``` 
#### The Solution: Reverse-Scoring

- To fix this so the math actually works, we must flip the scores of all negatively worded questions before doing any math.

- We want to transform negative Questions so that a 1 becomes a 5, a 2 becomes a 4, and so on.

- The Magic Formula: $6 - \text{Score}$

- Since the ratings only go from 1 to 5, subtracting any rating from 6 flips it perfectly.
```
```
#### Why 6 - score? What is 6 exactly?
- 6 is not a magic number; it is mathematically derived directly from the Minimum and Maximum values of the rating scale.

- The general universal rule to flip any Likert rating scale is: $$\text{Flipping Constant} = \text{Minimum Scale Value} + \text{Maximum Scale Value}$$
- Because our OCEAN survey uses a 1 to 5 scale:$$\text{Flipping Constant} = \mathbf{1} + \mathbf{5} = \mathbf{6}$$

#### 5.6.1 Finding the reverse columns
---

In [300]:
reverse_cols = []

for trait in ['EXT' , 'EST' , 'AGR' , 'CSN' , 'OPN']:

    # choosing the benchmark question(positive statement) to compare others with reference to that.
    if trait == "AGR":
        benchmark_item = "AGR2" # AGR1 is negative ("little concern"), so we use AGR

    else:
        benchmark_item = f'{trait}{1}'
   
    # Loop 2: Check questions 1 through 10 one by one
    for i in range(1 , 11):
        question_name = f'{trait}{i}'

        correlation_val = df_clean[question_name].corr(df_clean[benchmark_item])

        if correlation_val < 0:
            reverse_cols.append(question_name)

print("Reverse columns are :" , reverse_cols)
print("Total reversed columns found:", len(reverse_cols))


Reverse columns are : ['EXT2', 'EXT4', 'EXT6', 'EXT8', 'EXT10', 'EST2', 'EST4', 'AGR1', 'AGR3', 'AGR5', 'AGR7', 'CSN2', 'CSN4', 'CSN6', 'CSN8', 'OPN2', 'OPN4', 'OPN6']
Total reversed columns found: 18


#### 5.6.2  Apply scale flipping (6 - score)
----

In [303]:

df_clean[reverse_cols] = 6 - df_clean[reverse_cols]
print("Scale flipping complete!")

Scale flipping complete!


### 5.7 Aggregate the 10 Questions into 5 Trait Scores
------

- Right now our dataset has 50 separate columns (EXT1 through OPN10).

- If we feed 50 individual questions into a machine learning algorithm, it struggles with the "curse of dimensionality" that means too many individual noisy inputs. Psychologists designed these 50 questions to measure just 5 core dimensions of human personality:
  
    - Extraversion (EXT)

    - Emotional Stability / Neuroticism (EST)

    - Agreeableness (AGR)

    - Conscientiousness (CSN)

    - Openness (OPN

- By taking the average of the 10 questions for each trait, we compress 50 raw survey answers into 5 clean, high-level summary scores per person.

In [311]:

for trait in ['EXT' , 'EST' , 'AGR' , 'CSN' , 'OPN']:

        trait_cols = []
        for i in range(1 , 11):
            trait_cols.append(f'{trait}{i}')

        #finding the average of the 10 questions for each trait
        df_clean[f'{trait}_score'] = df_clean[trait_cols].mean(axis = 1)

# Inspect the newly created trait score columns
trait_score_cols = ['EXT_score', 'EST_score', 'AGR_score', 'CSN_score', 'OPN_score']

print("Aggregated Trait Scores (First 5 Rows):")

print(df_clean[trait_score_cols].head())

Aggregated Trait Scores (First 5 Rows):
   EXT_score  EST_score  AGR_score  CSN_score  OPN_score
0        4.6        2.4        3.9        3.2        4.5
1        2.0        2.5        4.4        3.7        3.5
2        2.5        2.6        4.2        3.4        4.1
3        2.6        2.9        3.8        2.5        3.9
4        2.9        1.9        4.6        4.8        4.8


---
## 6. Feature Scaling
---

### 1. What is Feature Scaling?
   
Feature Scaling is a data preprocessing technique used to bring independent features (variables) onto a similar numerical scale without distorting the underlying differences or shapes of their distributions.


### 2. Why Do We Need Feature Scaling here?

Imagine two questions on a $1\text{ to }5$ survey:

#### 1. Question A: "Do you like ice cream?"

 - Almost everyone answers 4 or 5.

 - The scores are tightly squished between 4.0 and 5.0. The spread (variance) is tiny here.
 
#### 2. Question B: "Are you comfortable speaking in front of 500 people?"

   - Answers are scattered all over the place—some people say 1, others say 3, others say 5.

   - The scores spread wildly across 1.0 to 5.0. The spread (variance) is massive.

On paper, both questions use $1\text{ to }5$. But in reality, Question B stretches out across a huge distance, while Question A is packed into a tiny corner.
```
```
### 3. Why K-Means Gets Fooled ?

- K-Means gets fooled without scaling because it measures similarity using pure straight-line distance (Euclidean distance) to the cluster flagpoles, and raw features with larger spreads act like mathematical bullies in the distance formula.
  
- If one trait spreads out wider than another in human responses, subtracting a person's score from a flagpole score yields a much larger number for that wide-spread trait, tricking K-Means into ignoring all other traits.
```
```  
#### Imagine K-Means places a Flagpole at the population baseline (average scores):

- Flagpole 0 Coordinates: [AGR = 3.8, EST = 3.0]

- Now look at a single person in your dataset:

-  Respondent 1: [AGR = 4.8, EST = 1.0]

- K-Means calculates how far Respondent 1 is from Flagpole 0 using subtraction and squaring:
```
```
#### 1. WITHOUT Scaling (Raw 1–5 Scores)
- Agreeableness Difference ($\Delta\text{AGR}$):$$4.8 - 3.8 = 1.0 \implies 1.0^2 = \mathbf{1.0}$$
- Emotional Stability Difference ($\Delta\text{EST}$):$$1.0 - 3.0 = -2.0 \implies (-2.0)^2 = \mathbf{4.0}$$$$\text{Distance} = \sqrt{1.0 + 4.0} = \sqrt{5.0}$$
  
- Why K-Means is Fooled:Inside the square root ($\sqrt{1.0 + 4.0}$), Emotional Stability makes up 80% of the total distance value ($4.0$ out of $5.0$). When K-Means shifts Flagpole 0 closer to this person, it will move 4 times further along the Emotional Stability axis than the Agreeableness axis. It gets tricked into creating clusters based mostly on EST, treating AGR like invisible noise.
```
```
#### 2. WITH Scaling (StandardScaler)
- StandardScaler rescales the features based on how humans actually answered:
  
- AGR has a tight spread ($\sigma \approx 0.5$). A $1.0$ raw shift = $+2.0$ Standard Deviations.
- EST has a wide spread ($\sigma \approx 1.0$). A $2.0$ raw shift = $-2.0$ Standard Deviations.

- Now look at the calculation from Respondent 1 to Flagpole 0 ([0, 0]):
- Scaled Agreeableness Difference:$$2.0 - 0.0 = 2.0 \implies 2.0^2 = \mathbf{4.0}$$

- Scaled Emotional Stability Difference:$$-2.0 - 0.0 = -2.0 \implies (-2.0)^2 = \mathbf{4.0}
$$$$\text{Distance} = \sqrt{4.0 + 4.0} = \sqrt{8.0}$$
```
```
- How Scaling Fixes It:Inside the square root ($\sqrt{4.0 + 4.0}$), both traits contribute exactly 50% ($4.0$). Because both traits represent equal physical deviation from the human average, the flagpole moves equally toward both traits.

In [370]:
from sklearn.preprocessing import StandardScaler

trait_score_cols = ['EXT_score', 'EST_score', 'AGR_score', 'CSN_score', 'OPN_score']

scaler =  StandardScaler()

X_scaled = scaler.fit_transform(df_clean[trait_score_cols])

print("Feature Scaling Complete!")

print(f"Shape of X_scaled array: {X_scaled.shape}")

print("\nFirst 3 rows of scaled data (Mean ~ 0, Std ~ 1):")


print(X_scaled[:3])




Feature Scaling Complete!
Shape of X_scaled array: (85393, 5)

First 3 rows of scaled data (Mean ~ 0, Std ~ 1):
[[ 1.81549552 -0.79559968  0.21572777 -0.14237234  1.05271258]
 [-1.04409091 -0.67668384  0.90509339  0.54284089 -0.57768263]
 [-0.49417045 -0.55776799  0.62934714  0.13171295  0.4005545 ]]


---
## 7. K-Means Model Fitting and Evaluation
---

### 7.1 Model fitting
----

In [426]:
from sklearn.cluster import KMeans

# 1. Initialize and fit K-Means model with K=5
# n_init = How many times the algorithm retries to guarantee a good starting position.
model = KMeans( n_clusters = 5 , random_state = 42 , n_init = 10)

                                                            
# 2. Fit model and predict cluster assignments for each sample
cluster_labels = model.fit_predict(X_scaled)


# 3. Save predicted clusters back to the main DataFrame
df_clean['Cluster'] = cluster_labels

print("K-Means Model Successfully Fitted!")
print(f"Cluster assignment count:\n{df_clean['Cluster'].value_counts().sort_index()}")


K-Means Model Successfully Fitted!
Cluster assignment count:
Cluster
0    21126
1    17961
2    16791
3    12260
4    17255
Name: count, dtype: int64


In [403]:
df_clean.head()


,EXT1,EXT2,EXT3,EXT4,EXT5,EXT6,EXT7,EXT8,EXT9,EXT10,...,OPN8,OPN9,OPN10,EXT_score,EST_score,AGR_score,CSN_score,OPN_score,Clusters,Cluster
0,4.0,5.0,5.0,4.0,5.0,5.0,5.0,4.0,4.0,5.0,...,3.0,4.0,5.0,4.6,2.4,3.9,3.2,4.5,2,2
1,3.0,1.0,3.0,2.0,3.0,3.0,2.0,1.0,1.0,1.0,...,2.0,5.0,3.0,2.0,2.5,4.4,3.7,3.5,0,0
2,2.0,3.0,4.0,2.0,3.0,4.0,1.0,3.0,2.0,1.0,...,3.0,4.0,4.0,2.5,2.6,4.2,3.4,4.1,0,0
3,2.0,4.0,2.0,3.0,4.0,4.0,2.0,2.0,1.0,2.0,...,4.0,3.0,3.0,2.6,2.9,3.8,2.5,3.9,2,2
4,3.0,3.0,3.0,3.0,5.0,3.0,3.0,1.0,3.0,2.0,...,3.0,5.0,5.0,2.9,1.9,4.6,4.8,4.8,0,0


In [405]:
df_clean = df_clean.drop(columns = ['Clusters']) #deleting extra column created by mistake

In [407]:
df_clean.head()

,EXT1,EXT2,EXT3,EXT4,EXT5,EXT6,EXT7,EXT8,EXT9,EXT10,...,OPN7,OPN8,OPN9,OPN10,EXT_score,EST_score,AGR_score,CSN_score,OPN_score,Cluster
0,4.0,5.0,5.0,4.0,5.0,5.0,5.0,4.0,4.0,5.0,...,5.0,3.0,4.0,5.0,4.6,2.4,3.9,3.2,4.5,2
1,3.0,1.0,3.0,2.0,3.0,3.0,2.0,1.0,1.0,1.0,...,4.0,2.0,5.0,3.0,2.0,2.5,4.4,3.7,3.5,0
2,2.0,3.0,4.0,2.0,3.0,4.0,1.0,3.0,2.0,1.0,...,5.0,3.0,4.0,4.0,2.5,2.6,4.2,3.4,4.1,0
3,2.0,4.0,2.0,3.0,4.0,4.0,2.0,2.0,1.0,2.0,...,4.0,4.0,3.0,3.0,2.6,2.9,3.8,2.5,3.9,2
4,3.0,3.0,3.0,3.0,5.0,3.0,3.0,1.0,3.0,2.0,...,5.0,3.0,5.0,5.0,2.9,1.9,4.6,4.8,4.8,0


### 7.2 Model Evaluation
----

#### 1. The Intuition Behind Model Evaluation

why we evaluate K-Means ?

- K-Means is a "dumb" algorithm in one specific way: it will always group our data into K clusters if we tell it to, even if the data is completely random noise.

- If we ask K-Means for 5 clusters, it will draw 5 boundaries, whether those boundaries make sense or not.
-  Evaluation is the quality check to prove those 5 boundaries represent real, distinct groups in high-dimensional space.
```
```
#### 2. Silhouette Score
- To check if our clusters are clean, we use a single key metric: the Silhouette Score.
  
- For every single person (data point) in your dataset, the Silhouette Score asks two simple questions:
  
  - Cohesion (a): "How close am I to people in my OWN cluster?" (Smaller distance = better)
    
  - Separation (b): "How far am I from people in the NEAREST OTHER cluster?" (Larger distance = better)
  - It combines these into a single balance score for that point:$$\text{Silhouette Score} = \frac{b - a}{\max(a, b)}$$

#### 3. How to Interpret the Output?

- The Silhouette Score gives you a number between -1.0 and +1.0:
  
  - **Near +1.0 (Isolated Groups):** Perfect separation. Points are tightly packed with their own group and far away from others.
    
  - **Around 0.0 (Overlapping / Fuzzy Boundaries):** Overlapping points. Data points sit right on the boundary between two clusters.
  - **Near -1.0(Wrong Assignment)::** Incorrect assignment. Points were placed in the wrong cluster.
```
```
#### 4. Code explanation 
- silhouette_score(...): The function measures how well each data point fits into its assigned group versus the neighboring group.
  
- X_scaled: our standardized feature matrix (Z-scores). It ensures distances along all 5 Big Five dimensions are measured on an equal scale.
- model.labels_: The list of cluster assignments (0, 1, 2, 3, or 4) generated for each participant by K-Means.
- sample_size=3000: Instead of building a massive distance matrix in memory that crashes RAM, it evaluates a random sample of 3,000 participants to compute an accurate estimate.
- random_state=42: Locks the random seed so our sub-sampling produces identical, reproducible results every time we execute the cell.
- model.inertia_: The total squared distance of every point to its assigned cluster center (centroid). Lower numbers mean tighter clusters.

In [480]:
from sklearn.metrics import silhouette_score

# Pass sample_size to prevent allocating huge distance matrices in RAM
silhouette_avg = silhouette_score(X_scaled, model.labels_, sample_size=3000, random_state=42)
inertia_val = model.inertia_

print("--- Model Quality Metrics (K = 5) ---")
print(f"Silhouette Score: {silhouette_avg:.2f}")
print(f"Inertia:          {inertia_val:.2f}")

--- Model Quality Metrics (K = 5) ---
Silhouette Score: 0.16
Inertia:          248665.13


#### Observations and interpretation:
----
A score near ~0.16 is expected for Behavioral Data. To understand why a score of 0.16 is completely valid, we compare physical categories with human behavior:

#### Physical Data (Discrete Objects)
Think of grouping Bicycles vs. Cargo Ships.
- There are no "semi-cargo bicycles" bridging the gap.
  
- Physical data forms isolated, hard-edged islands in space.
- Silhouette Score: High ($+0.60 \text{ to } +0.90$).
```
```
#### Behavioral Data (Human Psychology)
Think of grouping human personalities across 5 traits:

- People are not manufactured in isolated batches; personality exists along continuous spectrums.

- There are no empty gaps between human types—most people lie along the middle ground (e.g., ambiverts).

       

----
## 8. Defining Personality Archetypes & Group Profiles
---

- Up until now, our model only understands geometry, it sees point clouds, distances, and Z-score vectors. It gave us cluster tags like 0, 1, 2, 3, and 4. But in the real world, a product manager, VP of Engineering, or behavioral researcher cannot build anything with Cluster 2.
  
- Section 8 is the translation bridge. It takes abstract, N-dimensional space and translates it into distinct human behavioral profiles (Archetypes).

```
```
#### Why do we need Section 8?

- In supervised learning, our model outputs something directly readable: House Price = $450,000 or Spam = True.

- In unsupervised learning, there is no built-in label. The algorithm groups people based on proximity, but it doesn't know WHO it grouped. It’s like sorting a huge crowd into five different rooms based on how similarly they dress, but never looking inside the rooms to see what style each room actually contains.

- Section 8 is us opening those five doors, measuring the average trait vector of each room, and giving each group a meaningful human identity (e.g., "The Resilient Innovator" or "The Cautious Planner").
```
```

#### The Metric: Deciphering the Z-Score Language
- Before we look at the code, let's build the intuition for the unit of measurement we use: Z-scores.
  
- When you ran StandardScaler() in Section 6, you normalized all your traits to have a Mean ($\mu$) of $0.0$ and a Standard Deviation ($\sigma$) of $1.0$. This creates a universal baseline

### 8.1 Extract Settled Centroid Coordinates
-------
- We pull the internal matrix of final centroid positions (kmeans.cluster_centers_) into a labeled Pandas DataFrame.

In [448]:
import pandas as pd

# Define trait names in the EXACT order used during scaling/fitting
trait_names = ['Extraversion', 'Emotional_Stability', 'Agreeableness', 'Conscientiousness', 'Openness']

# Extract settled centroid coordinates into a structured DataFrame
archetypes_df = pd.DataFrame(
    model.cluster_centers_,
    columns = trait_names,
    index = [f'cluster{i}' for i in range(5)]
)

print("--- Settled Centroid Coordinates (Z-Scores) ---")
display(archetypes_df.round(2))

--- Settled Centroid Coordinates (Z-Scores) ---


,Extraversion,Emotional_Stability,Agreeableness,Conscientiousness,Openness
cluster0,0.56,-0.81,0.61,0.94,0.29
cluster1,-0.42,0.20,-0.31,-0.26,-1.32
cluster2,0.91,-0.00,0.36,-0.78,0.41
cluster3,-0.68,-0.19,-1.52,-0.01,0.33
cluster4,-0.65,0.92,0.31,-0.11,0.39


### 8.2 Persona Decoding & Mapping
-----
#### 1. Setting the Threshold Rules ($Z$-score Evaluation):
   
- Values $> +0.50$: Strongly elevated trait expression (dominant characteristics).
  
- Values between $-0.50$ and $+0.50$: Baseline or average trait expression.
- Values $< -0.50$: Suppressed or low trait expression.

#### 2. Identifying Peaks and Valleys:
Analyzing each row of archetypes_df to isolate its extreme values. For example, in our matrix:

- Cluster 0: High Conscientiousness ($+0.94$) paired with Low Emotional Stability ($-0.81$).
- Cluster 3: Extremely low Agreeableness ($-1.52$) paired with low Extraversion ($-0.68$).

#### 3. Assigning Business/Psychological Labels: 
Giving each cluster a representative name (such as "The Anxious Achiever" or "The Calm Anchor") based on how those 5 traits interact.
```
```
We translate the 5 settled centroid vectors into clear human behavioral profiles:

1. **Cluster 0 - "The Anxious Achiever"**
   * High Conscientiousness (+0.94) & Agreeableness (+0.61), Low Emotional Stability (-0.81).
   * Driven, organized, and cooperative, but prone to high stress.

2. **Cluster 1 - "The Traditionalist"**
   * Significantly Low Openness (-1.32) & Low Extraversion (-0.42).
   * Reserved; prefers familiar routines over abstract or novel ideas.

3. **Cluster 2 - "The Spontaneous Socializer"**
   * High Extraversion (+0.91), Low Conscientiousness (-0.78).
   * Outgoing and energetic, but lacks long-term planning/structure.

4. **Cluster 3 - "The Independent Skeptic"**
   * Very Low Agreeableness (-1.52) & Low Extraversion (-0.68).
   * Detached and highly self-reliant; operates independently of group norms.

5. **Cluster 4 - "The Calm Anchor"**
   * High Emotional Stability (+0.92), Low Extraversion (-0.65).
   * Introverted, yet steady, grounded, and resilient under pressure.

### 8.3 Mapping Personas Back to the Dataset
-----


In [495]:
# Create a dictionary mapping cluster numbers to human persona names
persona_map = {
    0: 'Anxious Achiever',
    1: 'Traditionalist',
    2: 'Spontaneous Socializer',
    3: 'Independent Skeptic',
    4: 'Calm Anchor'
}

# Assign named persona labels to your original dataset
df_clean['Persona'] = df_clean['Cluster'].map(persona_map)

print("--- Persona Distribution Count ---")
print(df_clean['Persona'].value_counts())

--- Persona Distribution Count ---
Persona
Anxious Achiever          21126
Traditionalist            17961
Calm Anchor               17255
Spontaneous Socializer    16791
Independent Skeptic       12260
Name: count, dtype: int64
